# 📄 Generador de Facturas - Dataset Completo

Este notebook genera facturas en formato dataset con:
- ✅ Estructura organizada (anotaciones/, facturas_procesadas/, rechazadas/, reportes/)
- ✅ JSONs con metadata completa
- ✅ PDFs sincronizados (mismo nombre)
- ✅ Guardado automático en Google Drive

## 📋 Paso 1: Configuración

In [ ]:
# ============================================================================
# CONFIGURACIÓN - MODIFICA ESTOS VALORES
# ============================================================================

# Cantidad de facturas a generar
CANTIDAD_FACTURAS = 10

# Ruta en Google Drive donde guardar (sin /content/drive/MyDrive/)
# Ejemplos:
#   "Facturas_Generadas"  → /content/drive/MyDrive/Facturas_Generadas/
#   "Proyectos/Facturas" → /content/drive/MyDrive/Proyectos/Facturas/
RUTA_DRIVE = "Facturas_Generadas"

# Generar PDFs (True) o solo JSONs (False)
# Nota: PDFs son más lentos pero más completos
GENERAR_PDFS = True

# ============================================================================

print("✅ Configuración cargada:")
print(f"   • Facturas a generar: {CANTIDAD_FACTURAS}")
print(f"   • Ruta en Drive: /content/drive/MyDrive/{RUTA_DRIVE}/")
print(f"   • Generar PDFs: {'Sí' if GENERAR_PDFS else 'No (solo JSONs)'}")

## 📁 Paso 2: Montar Google Drive

In [ ]:
from google.colab import drive
import os

# Montar Drive
drive.mount('/content/drive')

# Crear carpeta si no existe
ruta_completa = f"/content/drive/MyDrive/{RUTA_DRIVE}"
os.makedirs(ruta_completa, exist_ok=True)

print(f"\n✅ Drive montado correctamente")
print(f"✅ Carpeta lista: {ruta_completa}")

## 🔽 Paso 3: Clonar Repositorio e Instalar Dependencias

In [ ]:
# Clonar repositorio
!git clone https://github.com/GynoRomeroPrado/Creador-De-Factura.git
%cd Creador-De-Factura

# Instalar dependencias
!pip install -q reportlab python-dateutil faker

print("\n✅ Repositorio clonado e instalado")

## 🚀 Paso 4: Generar Facturas

In [ ]:
import sys
from datetime import datetime

# Importar módulos
sys.path.insert(0, 'src')
from src.generator import FacturaGenerator
from src.dataset_exporter import DatasetExporter
from src.pdf_creator import PDFFactura

print("=" * 70)
print(f"GENERANDO {CANTIDAD_FACTURAS} FACTURAS".center(70))
print("=" * 70)

# Crear exportador con ruta en Drive
exporter = DatasetExporter(base_dir=ruta_completa)
dataset_dir = exporter.crear_dataset()

print(f"\n📁 Dataset creado: {os.path.basename(dataset_dir)}")
print(f"   Ruta completa: {dataset_dir}")
print(f"\n📂 Estructura:")
print(f"   ├── anotaciones/")
print(f"   ├── facturas_procesadas/")
print(f"   ├── rechazadas/")
print(f"   └── reportes/")

# Generadores
factura_gen = FacturaGenerator()
pdf_gen = PDFFactura() if GENERAR_PDFS else None

# Generar facturas
print(f"\n🔄 Generando facturas...\n")

# Incluir facturas grandes cada 5 (20% del total tienen muchos items)
tipos_factura = ['general', 'hotel', 'seguro', 'con_descuento', 'compra_grande']
facturas_por_tipo = {tipo: 0 for tipo in tipos_factura}

for i in range(CANTIDAD_FACTURAS):
    # Balancear tipos
    tipo = tipos_factura[i % len(tipos_factura)]
    facturas_por_tipo[tipo] += 1
    
    # Generar factura
    factura = factura_gen.generar_factura(tipo_factura=tipo)
    
    # Generar PDF si está habilitado
    pdf_content = None
    if pdf_gen:
        try:
            pdf_path_temp = f"/tmp/temp_{factura['numero_factura']}.pdf"
            pdf_gen.crear_factura(factura, pdf_path_temp)
            with open(pdf_path_temp, 'rb') as f:
                pdf_content = f.read()
            os.remove(pdf_path_temp)
        except Exception as e:
            print(f"   ⚠️  Error generando PDF: {e}")
    
    # Exportar
    json_path, pdf_path = exporter.exportar_factura(factura, pdf_content)
    
    # Mostrar progreso
    serie = factura['numero_factura']
    emisor = factura['emisor']['razon_social'][:35]
    num_items = len(factura['items'])
    total = factura['simbolo_moneda'] + f"{factura['total']:,.2f}"
    status = "✅" if pdf_content else "📄"
    
    # Indicar si es factura grande (múltiples páginas)
    items_info = f"({num_items} items)" if num_items > 20 else ""
    
    print(f"   {status} [{i+1:3d}/{CANTIDAD_FACTURAS}] {serie} | {emisor:30s} | {total:>12s} {items_info}")

# Estadísticas
print("\n" + "=" * 70)
print("GENERACIÓN COMPLETADA".center(70))
print("=" * 70)

stats = exporter.obtener_estadisticas()
print(f"\n📊 Estadísticas:")
print(f"   • Dataset: {stats['dataset']}")
print(f"   • Anotaciones (JSON): {stats['anotaciones']}")
print(f"   • PDFs generados: {stats['pdfs']}")

print(f"\n📋 Facturas por tipo:")
for tipo, cant in facturas_por_tipo.items():
    tipo_nombre = tipo.replace('_', ' ').capitalize()
    print(f"   • {tipo_nombre:20s}: {cant:3d}")

print(f"\n💡 Nota: Las facturas 'Compra grande' tienen 30-50 items y múltiples páginas")
print(f"\n✅ Dataset guardado en Drive:")
print(f"   {dataset_dir}")

## ✔️ Paso 5: Verificar Archivos Generados

In [ ]:
import json

# Listar archivos
anotaciones_dir = os.path.join(dataset_dir, "anotaciones")
procesadas_dir = os.path.join(dataset_dir, "facturas_procesadas")

jsons = sorted([f for f in os.listdir(anotaciones_dir) if f.endswith('.json')])
pdfs = sorted([f for f in os.listdir(procesadas_dir) if f.endswith('.pdf')])

print("=" * 70)
print("VERIFICACIÓN DE ARCHIVOS".center(70))
print("=" * 70)

print(f"\n📄 JSONs en anotaciones/ ({len(jsons)}):")
for i, json_file in enumerate(jsons[:5], 1):
    print(f"   {i}. {json_file}")
if len(jsons) > 5:
    print(f"   ... y {len(jsons)-5} más")

print(f"\n📑 PDFs en facturas_procesadas/ ({len(pdfs)}):")
for i, pdf_file in enumerate(pdfs[:5], 1):
    print(f"   {i}. {pdf_file}")
if len(pdfs) > 5:
    print(f"   ... y {len(pdfs)-5} más")

# Verificar sincronización
print(f"\n🔗 Verificación de sincronización:")
sincronizados = sum(1 for j in jsons if j.replace('.json', '.pdf') in pdfs)
print(f"   • Archivos sincronizados (JSON ↔ PDF): {sincronizados}/{len(jsons)}")

if sincronizados == len(jsons):
    print("   ✅ Todos los archivos están sincronizados")
else:
    print(f"   ⚠️  Faltan {len(jsons) - sincronizados} PDFs")

# Mostrar ejemplo de JSON
if jsons:
    print(f"\n📋 Ejemplo de JSON ({jsons[0]}):")
    with open(os.path.join(anotaciones_dir, jsons[0]), 'r') as f:
        ejemplo = json.load(f)
    
    print(f"   • Serie: {ejemplo['serie_completa']}")
    print(f"   • Emisor: {ejemplo['emisor_razon_social']}")
    print(f"   • Receptor: {ejemplo['receptor_razon_social']}")
    print(f"   • Total: {ejemplo['importe_total']} {ejemplo['moneda']}")
    print(f"   • Condición: {ejemplo['condicion_pago']}")

## 📍 Paso 6: Mostrar Ruta Final en Drive

In [ ]:
print("=" * 70)
print("UBICACIÓN EN GOOGLE DRIVE".center(70))
print("=" * 70)

dataset_name = os.path.basename(dataset_dir)

print(f"\n📁 Tu dataset está guardado en:")
print(f"\n   Google Drive > Mi unidad > {RUTA_DRIVE} > {dataset_name}")
print(f"\n   Ruta completa: /content/drive/MyDrive/{RUTA_DRIVE}/{dataset_name}/")

print(f"\n📂 Contiene:")
print(f"   ├── anotaciones/         ({len(jsons)} JSONs)")
print(f"   ├── facturas_procesadas/ ({len(pdfs)} PDFs)")
print(f"   ├── rechazadas/          (vacía)")
print(f"   └── reportes/            (vacía)")

print(f"\n✅ ¡Listo! Verifica tu Drive para ver las facturas generadas.")